<a href="https://colab.research.google.com/github/RM-Menaka/FUTURE_ML_01/blob/main/SampleSuperstore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Import the libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

Load datasets

In [ ]:
df = pd.read_csv('/content/sample_data/SampleSuperstore.csv', encoding='ISO-8859-1')
print(df.head())

   Row ID        Order ID  Order Date   Ship Date       Ship Mode Customer ID  \
0       1  CA-2016-152156   11/8/2016  11/11/2016    Second Class    CG-12520   
1       2  CA-2016-152156   11/8/2016  11/11/2016    Second Class    CG-12520   
2       3  CA-2016-138688   6/12/2016   6/16/2016    Second Class    DV-13045   
3       4  US-2015-108966  10/11/2015  10/18/2015  Standard Class    SO-20335   
4       5  US-2015-108966  10/11/2015  10/18/2015  Standard Class    SO-20335   

     Customer Name    Segment        Country             City  ...  \
0      Claire Gute   Consumer  United States        Henderson  ...   
1      Claire Gute   Consumer  United States        Henderson  ...   
2  Darrin Van Huff  Corporate  United States      Los Angeles  ...   
3   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   
4   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   

  Postal Code  Region       Product ID         Category Sub-Category  \
0       42420   Sout

Basic data cleaning

In [ ]:
print("\nMissing Values")
print(df.isnull().sum())

Remove missing values

In [ ]:
df=df.dropna()

Step 4: convert Date column

In [ ]:
df['Order Date']=pd.to_datetime(df['Order Date'])

Step 5: Create monthly sales data

In [ ]:
monthly_sales=df.groupby(df['Order Date'].dt.to_period('M'))['Sales'].sum().reset_index()

monthly_sales['Order Date']=monthly_sales['Order Date'].astype(str)

monthly_sales['Month_Number']=np.arange(len(monthly_sales))

print("\nMonthly Sales Data: ")
print(monthly_sales.head())

PREPARING FEATURES AND TARGETS

In [ ]:
x=monthly_sales[['Month_Number']]
y=monthly_sales['Sales']


FIT AND TRAIN THE MODEL

In [ ]:
model = LinearRegression()
model.fit(x, y)

PREDICT THE EXISTING SALES

In [ ]:
predictions=model.predict(x)

FORECAST FUTURE SALES

In [ ]:
future_months=pd.DataFrame({
    'Month_Number':np.arange(len(monthly_sales),len(monthly_sales)+12)
})

future_predictions=model.predict(future_months)
print(len(future_predictions))

CREATE FUTURE DATES

In [ ]:
last_date=pd.to_datetime(monthly_sales['Order Date'].iloc[-1])

future_dates = pd.date_range(
    start=last_date,
    periods=13,
    freq='ME'
)[1:]
print(len(future_dates))

future_results=pd.DataFrame({
    'Future Date':future_dates,
    'Predictions':future_predictions
})

print("\nFuture Results: ")
print(future_results)

MODEL EVALUATION

In [ ]:
mae=mean_absolute_error(y,predictions)
print('\n Mean Absilute Error: ',mae)


VISUALIZATIONS

In [ ]:
plt.figure(figsize=(12,6))

plt.plot(

         monthly_sales['Month_Number'],
         y,
         label="Actual Sales"
)

plt.plot(

         future_months['Month_Number'],

         future_predictions,
         label="Predicted Sales"
)

plt.xlabel('Month Number')
plt.ylabel('Sales')
plt.title('Monthly Sales Forecast')
plt.legend()

plt.show()



EXPORT FORECAST RESULTS

In [ ]:
future_results.to_csv(

    'forecast_results.csv',
    index=False
)